In [1]:
!pip install python-telegram-bot langchain-openai langchain-core numpy joblib nest_asyncio
!pip install python-dotenv

In [2]:
import logging
import numpy as np
import joblib
import nest_asyncio
import pandas as pd
from telegram import Update, ReplyKeyboardMarkup, ReplyKeyboardRemove
from telegram.ext import (
    Application,
    CommandHandler,
    MessageHandler,
    ConversationHandler,
    filters,
    ContextTypes,
)
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv
import os
nest_asyncio.apply()

<h3> Настройка токенов

In [4]:
TELEGRAM_TOKEN   = os.getenv("TELEGRAM_TOKEN")
OPENROUTER_TOKEN = os.getenv("OPENROUTER_TOKEN")
MODEL_PATH       = "heart_model.pkl"

<h3> Инициализация

In [3]:
logging.basicConfig(
    format="%(asctime)s | %(levelname)s | %(message)s",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

try:
    ml_model = joblib.load(MODEL_PATH)
    logger.info("✅ ML-модель загружена успешно.")
except FileNotFoundError:
    logger.warning(f"⚠️ Файл '{MODEL_PATH}' не найден. Используется заглушка.")
    ml_model = None

llm = ChatOpenAI(
    api_key=OPENROUTER_TOKEN,
    base_url="https://openrouter.ai/api/v1",
    model="deepseek/deepseek-chat-v3.1",
)

2026-04-08 01:13:43,084 | INFO | ✅ ML-модель загружена успешно.


<h3> System Promt

In [4]:
SYSTEM_PROMPT = SystemMessage(content="""Ты — вежливый и внимательный медицинский ассистент.
Твоя задача — на основе данных пользователя и результата ML-модели составить
персонализированный ответ на русском языке.

Правила:
1. Обращайся к пользователю уважительно и с заботой. Не используй имена, только уважаемый пациент или уважаемая пациентка (в зависимости от пола)
2. Чётко укажи уровень риска: ВЫСОКИЙ или НИЗКИЙ — в зависимости от предсказания модели.
3. Дай 3–5 конкретных рекомендаций по образу жизни, учитывая данные пользователя.
4. Всегда советуй обратиться к врачу — ты не заменяешь специалиста.
5. Не ставь диагнозов. Используй слова «риск», «вероятность».
6. Тон: тёплый, профессиональный, без запугивания.
7. Объём ответа: 150–250 слов.""")

<h3>Вспомогательные функции парсинга + шаги диалога

In [5]:
(
    AGE,
    GENDER,
    CHEST_PAIN,
    BLOOD_PRESSURE,
    CHOL,
    FBS,
    RESTECG,
    MAX_HR,
    EXANG,
    OLDPEAK,
    SLOPE,
    CA,
    THAL,
    HEART_CONDITION,
    HEART_DISEASE,
) = range(15)

CHEST_PAIN_LABELS = {
    0: "типичная стенокардия",
    1: "атипичная стенокардия",
    2: "неангинозная боль",
    3: "бессимптомная",
}
RESTECG_LABELS = {
    0: "норма",
    1: "аномалия ST-T волны",
    2: "гипертрофия левого желудочка",
}
SLOPE_LABELS = {
    0: "восходящий",
    1: "плоский",
    2: "нисходящий",
}
CA_LABELS = {
    0: "ни одного сосуда со значимым сужением.",
    1: "один сосуд со значимым сужением.",
    2: "два сосуда со значимым сужением.",
    3: "три сосуда поражены",
}
THAL_LABELS = {
    0: "норма",
    1: "фиксированный дефект",
    2: "обратимый дефект",
    3: "другой или неопределённый",
}

def parse_gender(text: str):
    t = text.strip().lower()
    if t in ("мужской", "мужчина", "муж", "м", "m", "male", "1"):
        return 1
    if t in ("женский", "женщина", "жен", "ж", "f", "female", "0"):
        return 0
    return None

def parse_yes_no(text: str):
    t = text.strip().lower()
    if t in ("да", "yes", "1", "есть", "имеется", "+"):
        return 1
    if t in ("нет", "no", "0", "отсутствует", "-"):
        return 0
    return None

<h3> ML логика

In [6]:
def prepare_features(data: dict) -> pd.DataFrame:
    column_order = [
        'age', 'trestbps', 'chol', 'thalach', 'oldpeak',
        'sex', 'fbs', 'exang',
        'cp', 'restecg', 'slope', 'ca', 'thal'
    ]
    features_dict = {
        'age':     data["age"],
        'trestbps': data["blood_pressure"],
        'chol':    data["chol"],
        'thalach': data["max_hr"],
        'oldpeak': data["oldpeak"],
        'sex':     data["sex"],
        'fbs':     data["fbs"],
        'exang':   data["exang"],
        'cp':      data["chest_pain"],
        'restecg': data["restecg"],
        'slope':   data["slope"],
        'ca':      data["ca"],
        'thal':    data["thal"],
    }
    df = pd.DataFrame([features_dict])
    return df[column_order]

def get_prediction(features: np.ndarray) -> int:
    if ml_model is None:
        return 1
    return int(ml_model.predict(features)[0])

<h3> LLM-логика

In [11]:


def build_user_message(data: dict, prediction: int) -> str:
    gender_str     = "мужчина" if data["sex"] == 1 else "женщина"
    chest_pain_str = CHEST_PAIN_LABELS.get(data["chest_pain"], "?")
    fbs_str        = "выше 120 мг/дл" if data["fbs"] else "в норме"
    restecg_str    = RESTECG_LABELS.get(data["restecg"], "?")
    exang_str      = "есть" if data["exang"] else "нет"
    slope_str      = SLOPE_LABELS.get(data["slope"], "?")
    ca_str         = CA_LABELS.get(data["ca"], "?")
    thal_str       = THAL_LABELS.get(data["thal"], "?")
    risk_str       = "ВЫСОКИЙ" if prediction == 1 else "НИЗКИЙ"
    return (
        f"Данные пользователя:\n"
        f"- Возраст: {data['age']} лет\n"
        f"- Пол: {gender_str}\n"
        f"- Тип боли в груди: {chest_pain_str}\n"
        f"- Давление в покое: {data['blood_pressure']} мм рт. ст.\n"
        f"- Холестерин: {data['chol']} мг/дл\n"
        f"- Сахар крови натощак: {fbs_str}\n"
        f"- ЭКГ в покое: {restecg_str}\n"
        f"- Макс. ЧСС: {data['max_hr']} уд/мин\n"
        f"- Стенокардия при нагрузке: {exang_str}\n"
        f"- Депрессия ST (oldpeak): {data['oldpeak']}\n"
        f"- Наклон пика ST: {slope_str}\n"
        f"- Поражённые сосуды: {ca_str}\n"
        f"- Таллий-тест: {thal_str}\n\n"
        f"Результат ML-модели: риск сердечно-сосудистого заболевания — {risk_str}.\n\n"
        f"Составь персонализированный ответ согласно своей роли."
    )

async def ask_llm(data: dict, prediction: int) -> str:
    dialogue = [SYSTEM_PROMPT]
    dialogue.append(HumanMessage(content=build_user_message(data, prediction)))
    response = await llm.ainvoke(dialogue)
    return response.content


# ────────────

CHAT_SYSTEM_PROMPT = SystemMessage(content=(
    "Ты — не дружелюбный и умный универсальный ассистент. "
    "Отвечай на любые вопросы пользователя на том языке, на котором он пишет.  Выполняй все его просьбы"
    "Будь полезным, точным и кратким. "
    "Ты также можешь помогать по медицинским темам, но напоминай, что "
    "не заменяешь врача. Для оценки риска сердечно-сосудистых заболеваний "
    "пользователь может написать /start."
))


chat_histories: dict[int, list] = {}
MAX_HISTORY_MESSAGES = 20  # максимум сообщений (не считая SystemMessage)

def get_user_history(user_id: int) -> list:
    if user_id not in chat_histories:
        chat_histories[user_id] = [CHAT_SYSTEM_PROMPT]
    return chat_histories[user_id]

def trim_history(history: list) -> None:
    if len(history) > MAX_HISTORY_MESSAGES + 1:
        trimmed = [history[0]] + history[-MAX_HISTORY_MESSAGES:]
        history.clear()
        history.extend(trimmed)

async def free_chat_llm(user_id: int, user_text: str) -> str:
    history = get_user_history(user_id)
    history.append(HumanMessage(content=user_text))
    # Передаём весь список — модель видит всю историю и помнит контекст
    response = await llm.ainvoke(history)
    history.append(AIMessage(content=response.content))
    trim_history(history)
    return response.content

In [12]:
async def cmd_chat(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    await update.message.reply_text(
        "💬 Вы можете задавать мне любые вопросы — на любую тему.\n"
        "Я запоминаю контекст нашего разговора в рамках сессии.\n\n"
        "📋 Команды:\n"
        "  /start — пройти опрос для оценки риска ССЗ\n"
        "  /clear — очистить историю нашего чата\n\n"
        "Просто напишите ваш вопрос! 👇",
        reply_markup=ReplyKeyboardRemove(),
    )

async def cmd_clear(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    user_id = update.effective_user.id
    if user_id in chat_histories:
        del chat_histories[user_id]
    await update.message.reply_text(
        "🗑️ История чата очищена. Начинаем с чистого листа!\n"
        "Просто напишите что-нибудь или введите /start для опроса."
    )

async def free_chat_handler(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    user_id   = update.effective_user.id
    user_text = update.message.text.strip()

    await context.bot.send_chat_action(
        chat_id=update.effective_chat.id,
        action="typing"
    )

    try:
        answer = await free_chat_llm(user_id, user_text)
    except Exception as e:
        logger.error(f"Ошибка free_chat LLM для user {user_id}: {e}")
        answer = "⚠️ Произошла ошибка при обращении к модели. Попробуйте позже."

    await update.message.reply_text(answer)

<h3> Обработчик бота

In [13]:
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    await update.message.reply_text(
        "👋 Здравствуйте! Я медицинский ассистент.\n\n"
        "Задам несколько вопросов и оценю ваш риск сердечно-сосудистых заболеваний "
        "с помощью ML-модели, после чего дам персональные рекомендации.\n\n"
        "⚠️ Это информационная оценка — не медицинский диагноз.\n"
        "Введите /cancel для отмены.\n"
        "💬 Вне опроса вы можете задавать любые вопросы — просто напишите!\n\n"
        "➡️ Сколько вам полных лет?",
        reply_markup=ReplyKeyboardRemove(),
    )
    return AGE

async def get_age(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    try:
        age = int(update.message.text.strip())
        if not 1 <= age <= 120:
            raise ValueError
        context.user_data["age"] = age
    except ValueError:
        await update.message.reply_text("❌ Введите корректный возраст (число от 1 до 120):")
        return AGE

    keyboard = [["Мужской", "Женский"]]
    await update.message.reply_text(
        "➡️ Укажите ваш пол:",
        reply_markup=ReplyKeyboardMarkup(keyboard, one_time_keyboard=True, resize_keyboard=True),
    )
    return GENDER

async def get_gender(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    gender = parse_gender(update.message.text)
    if gender is None:
        await update.message.reply_text("❌ Не распознал ответ. Напишите «мужчина» или «женщина»:")
        return GENDER
    context.user_data["sex"] = gender

    keyboard = [["0", "1", "2", "3"]]
    await update.message.reply_text(
        "➡️ Укажите тип боли в груди:\n\n"
        "0 — типичная стенокардия\n"
        "1 — атипичная стенокардия\n"
        "2 — неангинозная боль\n"
        "3 — бессимптомная (боли нет)",
        reply_markup=ReplyKeyboardMarkup(keyboard, one_time_keyboard=True, resize_keyboard=True),
    )
    return CHEST_PAIN

async def get_chest_pain(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    try:
        val = int(update.message.text.strip())
        if val not in (0, 1, 2, 3):
            raise ValueError
        context.user_data["chest_pain"] = val
    except ValueError:
        await update.message.reply_text("❌ Введите цифру от 0 до 3:")
        return CHEST_PAIN

    await update.message.reply_text(
        "➡️ Введите систолическое (верхнее) артериальное давление в покое.\nНапример: 120",
        reply_markup=ReplyKeyboardRemove(),
    )
    return BLOOD_PRESSURE

async def get_blood_pressure(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    try:
        val = int(update.message.text.strip())
        if not 60 <= val <= 250:
            raise ValueError
        context.user_data["blood_pressure"] = val
    except ValueError:
        await update.message.reply_text("❌ Введите число от 60 до 250:")
        return BLOOD_PRESSURE

    await update.message.reply_text("➡️ Введите уровень холестерина (мг/дл).\nНапример: 220")
    return CHOL

async def get_chol(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    try:
        val = int(update.message.text.strip())
        if not 100 <= val <= 600:
            raise ValueError
        context.user_data["chol"] = val
    except ValueError:
        await update.message.reply_text("❌ Введите число от 100 до 600:")
        return CHOL

    keyboard = [["Да", "Нет"]]
    await update.message.reply_text(
        "➡️ Уровень сахара крови натощак выше 120 мг/дл?",
        reply_markup=ReplyKeyboardMarkup(keyboard, one_time_keyboard=True, resize_keyboard=True),
    )
    return FBS

async def get_fbs(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    val = parse_yes_no(update.message.text)
    if val is None:
        await update.message.reply_text("❌ Напишите «да» или «нет»:")
        return FBS
    context.user_data["fbs"] = val

    keyboard = [["0", "1", "2"]]
    await update.message.reply_text(
        "➡️ Результат ЭКГ в покое:\n\n0 — норма\n1 — аномалия ST-T волны\n2 — гипертрофия левого желудочка",
        reply_markup=ReplyKeyboardMarkup(keyboard, one_time_keyboard=True, resize_keyboard=True),
    )
    return RESTECG

async def get_restecg(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    try:
        val = int(update.message.text.strip())
        if val not in (0, 1, 2):
            raise ValueError
        context.user_data["restecg"] = val
    except ValueError:
        await update.message.reply_text("❌ Введите 0, 1 или 2:")
        return RESTECG

    await update.message.reply_text(
        "➡️ Введите максимальную ЧСС при физической нагрузке (уд/мин).\nНапример: 150",
        reply_markup=ReplyKeyboardRemove(),
    )
    return MAX_HR

async def get_max_hr(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    try:
        val = int(update.message.text.strip())
        if not 60 <= val <= 250:
            raise ValueError
        context.user_data["max_hr"] = val
    except ValueError:
        await update.message.reply_text("❌ Введите число от 60 до 250:")
        return MAX_HR

    keyboard = [["Да", "Нет"]]
    await update.message.reply_text(
        "➡️ Возникала ли у вас стенокардия во время физической нагрузки?",
        reply_markup=ReplyKeyboardMarkup(keyboard, one_time_keyboard=True, resize_keyboard=True),
    )
    return EXANG

async def get_exang(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    val = parse_yes_no(update.message.text)
    if val is None:
        await update.message.reply_text("❌ Напишите «да» или «нет»:")
        return EXANG
    context.user_data["exang"] = val

    await update.message.reply_text(
        "➡️ Введите значение депрессии ST-сегмента (oldpeak).\n"
        "Спросите у врача или введите 0, если не знаете.\nНапример: 1.5",
        reply_markup=ReplyKeyboardRemove(),
    )
    return OLDPEAK

async def get_oldpeak(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    try:
        val = float(update.message.text.strip().replace(",", "."))
        if not 0 <= val <= 10:
            raise ValueError
        context.user_data["oldpeak"] = val
    except ValueError:
        await update.message.reply_text("❌ Введите число от 0 до 10 (например: 1.5 или 0):")
        return OLDPEAK

    keyboard = [["0", "1", "2"]]
    await update.message.reply_text(
        "➡️ Наклон пика ST-сегмента при нагрузке:\n\n0 — восходящий\n1 — плоский\n2 — нисходящий\n\nЕсли не знаете — введите 1.",
        reply_markup=ReplyKeyboardMarkup(keyboard, one_time_keyboard=True, resize_keyboard=True),
    )
    return SLOPE

async def get_slope(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    try:
        val = int(update.message.text.strip())
        if val not in (0, 1, 2):
            raise ValueError
        context.user_data["slope"] = val
    except ValueError:
        await update.message.reply_text("❌ Введите 0, 1 или 2:")
        return SLOPE

    keyboard = [["0", "1", "2", "3"]]
    await update.message.reply_text(
        "➡️ Количество крупных сосудов со значимым сужением:\n\n"
        "0 — ни одного\n1 — один\n2 — два\n3 — три\n\nЕсли не знаете — введите 0.",
        reply_markup=ReplyKeyboardMarkup(keyboard, one_time_keyboard=True, resize_keyboard=True),
    )
    return CA

async def get_ca(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    try:
        val = int(update.message.text.strip())
        if val not in (0, 1, 2, 3):
            raise ValueError
        context.user_data["ca"] = val
    except ValueError:
        await update.message.reply_text("❌ Введите цифру от 0 до 3:")
        return CA

    keyboard = [["0", "1", "2", "3"]]
    await update.message.reply_text(
        "➡️ Результат таллий-сцинтиграфии (thal):\n\n"
        "0 — норма\n1 — фиксированный дефект\n2 — обратимый дефект\n3 — другой / неопределённый\n\nЕсли не знаете — введите 0.",
        reply_markup=ReplyKeyboardMarkup(keyboard, one_time_keyboard=True, resize_keyboard=True),
    )
    return THAL

async def get_thal(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    try:
        val = int(update.message.text.strip())
        if val not in (0, 1, 2, 3):
            raise ValueError
        context.user_data["thal"] = val
    except ValueError:
        await update.message.reply_text("❌ Введите цифру от 0 до 3:")
        return THAL

    await update.message.reply_text("⏳ Анализирую данные, пожалуйста подождите...")

    user_data  = context.user_data
    features   = prepare_features(user_data)
    prediction = get_prediction(features)

    logger.info(f"User {update.effective_user.id} | {dict(user_data)} | prediction={prediction}")

    try:
        llm_answer = await ask_llm(user_data, prediction)
    except Exception as e:
        logger.error(f"Ошибка LLM: {e}")
        risk_label = "высоким" if prediction == 1 else "низким"
        llm_answer = (
            f"По оценке модели, риск сердечно-сосудистых заболеваний является {risk_label}.\n"
            "К сожалению, развёрнутые рекомендации временно недоступны. "
            "Пожалуйста, обратитесь к врачу."
        )

    risk_emoji = "🔴" if prediction == 1 else "🟢"
    risk_label = "ВЫСОКИЙ" if prediction == 1 else "НИЗКИЙ"
    header = f"{risk_emoji} Оценка модели: риск *{risk_label}*\n\n"

    await update.message.reply_text(header + llm_answer, parse_mode="Markdown")
    await update.message.reply_text("Для повторной оценки введите /start.")
    return ConversationHandler.END

async def cancel(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    await update.message.reply_text(
        "❌ Опрос отменён. Для повторного запуска введите /start.",
        reply_markup=ReplyKeyboardRemove(),
    )
    return ConversationHandler.END

<h3> Запуск

In [ ]:
def main() -> None:
    app = Application.builder().token(TELEGRAM_TOKEN).build()

    conv_handler = ConversationHandler(
        entry_points=[CommandHandler("start", start)],
        states={
            AGE:            [MessageHandler(filters.TEXT & ~filters.COMMAND, get_age)],
            GENDER:         [MessageHandler(filters.TEXT & ~filters.COMMAND, get_gender)],
            CHEST_PAIN:     [MessageHandler(filters.TEXT & ~filters.COMMAND, get_chest_pain)],
            BLOOD_PRESSURE: [MessageHandler(filters.TEXT & ~filters.COMMAND, get_blood_pressure)],
            CHOL:           [MessageHandler(filters.TEXT & ~filters.COMMAND, get_chol)],
            FBS:            [MessageHandler(filters.TEXT & ~filters.COMMAND, get_fbs)],
            RESTECG:        [MessageHandler(filters.TEXT & ~filters.COMMAND, get_restecg)],
            MAX_HR:         [MessageHandler(filters.TEXT & ~filters.COMMAND, get_max_hr)],
            EXANG:          [MessageHandler(filters.TEXT & ~filters.COMMAND, get_exang)],
            OLDPEAK:        [MessageHandler(filters.TEXT & ~filters.COMMAND, get_oldpeak)],
            SLOPE:          [MessageHandler(filters.TEXT & ~filters.COMMAND, get_slope)],
            CA:             [MessageHandler(filters.TEXT & ~filters.COMMAND, get_ca)],
            THAL:           [MessageHandler(filters.TEXT & ~filters.COMMAND, get_thal)],
        },
        fallbacks=[CommandHandler("cancel", cancel)],
    )

    # ConversationHandler первым — перехватывает сообщения во время опроса
    # free_chat_handler вторым — обрабатывает всё остальное
    app.add_handler(conv_handler)
    app.add_handler(CommandHandler("chat",  cmd_chat))
    app.add_handler(CommandHandler("clear", cmd_clear))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, free_chat_handler))

    logger.info("🤖 Бот запущен. Ожидание сообщений...")
    app.run_polling(allowed_updates=Update.ALL_TYPES)

main()

2026-04-08 01:17:00,395 | INFO | 🤖 Бот запущен. Ожидание сообщений...
2026-04-08 01:17:00,676 | INFO | HTTP Request: POST https://api.telegram.org/bot8619886332:AAEdtepT_Wcf9PzhMYBjZkecjShiSi-3ai4/getMe "HTTP/1.1 200 OK"
2026-04-08 01:17:00,755 | INFO | HTTP Request: POST https://api.telegram.org/bot8619886332:AAEdtepT_Wcf9PzhMYBjZkecjShiSi-3ai4/deleteWebhook "HTTP/1.1 200 OK"
2026-04-08 01:17:00,757 | INFO | Application started
2026-04-08 01:17:07,493 | INFO | HTTP Request: POST https://api.telegram.org/bot8619886332:AAEdtepT_Wcf9PzhMYBjZkecjShiSi-3ai4/getUpdates "HTTP/1.1 200 OK"
2026-04-08 01:17:07,704 | INFO | HTTP Request: POST https://api.telegram.org/bot8619886332:AAEdtepT_Wcf9PzhMYBjZkecjShiSi-3ai4/sendChatAction "HTTP/1.1 200 OK"
2026-04-08 01:17:10,952 | INFO | HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-08 01:17:17,579 | INFO | HTTP Request: POST https://api.telegram.org/bot8619886332:AAEdtepT_Wcf9PzhMYBjZkecjShiSi-3ai4/getUpdat